# 古籍批量自动标点 — Qwen3.5-4B Q4_K_M (llama-cpp 单流) 测试

目标：拿 shu-yuan-zaji 看 4B Q4_K_M 标点质量；速度也顺便记一下。

**模型**：`bartowski/Qwen_Qwen3.5-4B-GGUF` 的 Q4_K_M（~2.5 GB），单张 T4 够跑

**Accelerator**：GPU T4 x1 或 T4 x2 都行（不用 P100）

**已避坑**：abetlen 预编译 wheel（不现编） / 模型下 /tmp / fail-fast import 校验

## 0. 环境 + 选书

In [ ]:
import os

os.environ['HF_HOME'] = '/tmp/hf_cache'
os.environ['HF_HUB_CACHE'] = '/tmp/hf_cache/hub'

# P7 预切版（副账号 canhuiliphy 跑）：长段已本地切到 ≤350 字带 marker
BOOK = 'mingshilu-p7-prechunked'
PART = 1
PART_OF = 1
DATASET_SLUG = 'canhuiliphy/mingshilu-prechunked-p78'  # 副账号 dataset

REPO = 'bartowski/Qwen_Qwen3-30B-A3B-Instruct-2507-GGUF'
GGUF_PATTERN = 'Q4_K_M'

print('--- /kaggle/input/ ---')
!ls /kaggle/input/ 2>&1
print('--- 递归找 jsonl ---')
!find /kaggle/input -name "*.jsonl" 2>&1

IN_PATH = f'/kaggle/input/mingshilu-prechunked-p78/{BOOK}.jsonl'
if not os.path.exists(IN_PATH):
    import subprocess
    try:
        found = subprocess.check_output(['find', '/kaggle/input', '-name', f'{BOOK}.jsonl', '-type', 'f']).decode().strip().splitlines()
        if found:
            IN_PATH = found[0]
            print(f'\n自动定位到：{IN_PATH}')
    except subprocess.CalledProcessError:
        pass

SUFFIX = f'-p{PART}of{PART_OF}' if PART_OF > 1 else ''
OUT_PATH = f'/kaggle/working/{BOOK}{SUFFIX}-punctuated.jsonl'
CKPT     = f'/kaggle/working/{BOOK}{SUFFIX}-checkpoint.json'
print(f'\nbook={BOOK}  part={PART}/{PART_OF}\nIN_PATH={IN_PATH}\nOUT_PATH={OUT_PATH}')
!df -h /tmp /kaggle/working 2>/dev/null | head -5


## 1. GPU 检查（要 T4 即可，单卡或双卡都行）

In [ ]:
import subprocess
info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap', '--format=csv,noheader']).decode().strip()
print(info)
lines = info.splitlines()
N_GPU = len(lines)
caps = [float(l.split(',')[-1].strip()) for l in lines]
assert all(c >= 7.0 for c in caps), f'Pascal 不支持，右栏改 GPU T4。caps={caps}'
print(f'OK: {N_GPU} GPU(s)')

## 2. 装 llama-cpp-python（预编译 wheel）+ 校验

In [ ]:
# llama-cpp-python 安装：用 +cu local version 强迫 pip 选 cu wheel
# 原因：abetlen 的 cu wheel 命名是 llama_cpp_python==X.Y.Z+cuNNN；
#       PyPI vanilla 是 X.Y.Z（无 +cu 后缀）；
#       --extra-index-url 只是补充，pip 默认信 PyPI 的更"干净"版本。
#       用 ==X+cuNNN 强制锁后缀，pip 就必须去 abetlen 找。
import subprocess, os, time, re, importlib

def run_pip(cmd, attempts=3, sleep=30):
    for i in range(attempts):
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0: return True, ''
        err = (r.stderr or '') + (r.stdout or '')
        is_net = any(s in err for s in ['Temporary failure','NewConnectionError','ConnectTimeout','ReadTimeoutError'])
        if is_net and i < attempts-1:
            print(f'网络抖动，{sleep}s 后重试...')
            time.sleep(sleep)
        else:
            return False, err
    return False, 'exhausted'

assert run_pip(['pip','install','-q','--upgrade','huggingface_hub'])[0]

# 找 PyPI 当前最新版本
r = subprocess.run(['pip','index','versions','llama-cpp-python'], capture_output=True, text=True)
m = re.search(r'Available versions:\s*([\d.]+)', r.stdout)
ver = m.group(1) if m else '0.3.20'
print(f'PyPI 最新版本: {ver}')

# 尝试用 +cu local version 锁定 GPU wheel（cu126 → cu125 → cu124）
installed_via = None
for cu in ('cu126', 'cu125', 'cu124'):
    # 注意：版本号要带 +cu 后缀，pip 才会去 extra-index 找
    spec = f'llama-cpp-python=={ver}+{cu}'
    print(f'\n--- 尝试 {spec} ---')
    ok, err = run_pip([
        'pip','install', spec,
        '--extra-index-url', f'https://abetlen.github.io/llama-cpp-python/whl/{cu}',
        '--upgrade','--force-reinstall','--no-cache-dir',
    ])
    if not ok:
        print(f'装不上: {err[-500:]}')
        # 如果是版本不存在，试这个 cu 版本的最新 release
        if 'No matching distribution' in err:
            print(f'  {ver}+{cu} 不存在，跳过这档')
        continue
    try:
        if 'llama_cpp' in dir(): importlib.reload(llama_cpp)
        else: import llama_cpp
        if llama_cpp.llama_supports_gpu_offload():
            print(f'✅ {cu} wheel 装上且 GPU offload OK')
            installed_via = cu; break
        else:
            print(f'{cu} 装上但 GPU offload 失败（可能不是 cu wheel）')
    except Exception as e:
        print(f'{cu} import 失败: {e}')

if installed_via is None:
    print('\n--- 退一步：去掉 +cu 锁定，让 pip 自由从 cu126 索引拿（最新可用版本）---')
    ok, err = run_pip([
        'pip','install','llama-cpp-python',
        '--index-url','https://abetlen.github.io/llama-cpp-python/whl/cu126',
        '--extra-index-url','https://pypi.org/simple',
        '--upgrade','--force-reinstall','--no-cache-dir',
    ])
    if ok:
        try:
            if 'llama_cpp' in dir(): importlib.reload(llama_cpp)
            else: import llama_cpp
            if llama_cpp.llama_supports_gpu_offload():
                installed_via = 'cu126_index_url'
        except: pass

if installed_via is None:
    print('\n--- 最后兜底：源码编译 ---')
    os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on'
    os.environ['FORCE_CMAKE'] = '1'
    ok, err = run_pip([
        'pip','install','--upgrade','--force-reinstall','--no-cache-dir',
        '--retries','5','--timeout','120','-v',
        'llama-cpp-python','--no-binary','llama-cpp-python',
    ])
    if not ok:
        # 打印更长的错误信息以便排查
        print(f'\n源码编译失败完整日志（尾 3000 字符）:\n{err[-3000:]}')
        raise RuntimeError('llama-cpp-python 装不上，所有路径都失败')
    installed_via = 'source_build'

print(f'\n✅ llama-cpp-python via {installed_via}, version={llama_cpp.__version__}')
print(f'   GPU offload: {llama_cpp.llama_supports_gpu_offload()}')

assert run_pip(['pip','install','-q','opencc-python-reimplemented','yitizi'])[0]

In [ ]:
# 验证 llama_cpp 已装好且能用 GPU
import llama_cpp
print(f'llama_cpp version: {llama_cpp.__version__}')
print(f'CUDA offload: {llama_cpp.llama_supports_gpu_offload()}')
print(f'wheel 路径: {llama_cpp.__file__}')
assert llama_cpp.llama_supports_gpu_offload(), 'GPU 还是没编进 wheel'

## 3. 下载 GGUF 到 /tmp（~2.5 GB，1-3 min）

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
import time

api = HfApi()
files = api.list_repo_files(REPO)
matches = sorted([f for f in files if GGUF_PATTERN in f and f.endswith('.gguf')])
print('matching files:'); [print(f'  - {f}') for f in matches]
assert matches, f'no file matching {GGUF_PATTERN} in {REPO}'
GGUF_FILE = matches[0]

print(f'\ndownloading {GGUF_FILE}')
t0 = time.time()
model_path = hf_hub_download(repo_id=REPO, filename=GGUF_FILE)
print(f'done in {(time.time()-t0)/60:.1f} min → {model_path}')
print(f'size: {os.path.getsize(model_path)/(1024**3):.2f} GB')

## 4. 加载模型（单卡，4B 不需要 tensor split）

In [ ]:
from llama_cpp import Llama

# 30B-A3B Q5_K_M ~20GB → 必须 tensor split 到 T4 x2
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_batch=512,
    n_threads=4,
    n_gpu_layers=-1,
    tensor_split=[0.5, 0.5],
    main_gpu=0,
    flash_attn=False,
    verbose=False,
)
print('model loaded')

## 5. Prompt + 校验

In [ ]:
FEWSHOT_PROMPT = '''下面是给古文加中文标点的任务。

规则：
1. 保留所有原字，只在字间插入标点（，。：；？！、《》「」）。
2. 段落必须以句末符号结尾（。 ？ ！ 」 』 ）之一）；不要以 ， ： 、 ； 收尾。
3. 不要连续两个标点（除了 。」 ？」 ！」 这类引号闭合）。例如「，：」「，，」「。，」都是错的。
4. 「」必须成对出现，《》必须成对出现。开了引号一定要在合适位置闭合。
5. 句意为重，宁可让句子稍长一点，也不要把名字、官职、书名硬切成多段。

无标点：自古帝王之有天下其言行政治必有史臣纪载以垂鉴戒此古今之盛典朝廷之先务也
有标点：自古帝王之有天下，其言行政治，必有史臣纪载，以垂鉴戒，此古今之盛典，朝廷之先务也。

无标点：奉天门常朝御座后内官持一小扇金黄绢以裹之尝闻一老将军云非扇也其名卓影辟邪永乐间外国所进
有标点：奉天门常朝，御座后内官持一小扇，金黄绢以裹之。尝闻一老将军云：「非扇也，其名卓影辟邪，永乐间外国所进。」

无标点：本朝六卿之设虽祖周官而六部之名实沿唐制但唐之六部为尚书省之属曹
有标点：本朝六卿之设，虽祖周官，而六部之名实沿唐制。但唐之六部为尚书省之属曹。

无标点：吾乡布衣沈先生名璵字孟温洪武中其家坐累谪戍云南之金齒宣徳初归省坟墓乡人以其经学该愽留教子弟
有标点：吾乡布衣沈先生，名璵，字孟温。洪武中，其家坐累，谪戍云南之金齒。宣徳初，归省坟墓，乡人以其经学该愽，留教子弟。

无标点：{raw}
有标点：'''

PUNCT_SET = set(
    '，。：；？！「」、,.:;?!"《》〈〉（）()—…——·・•‧『』〔〕\\/／'
)
WS_SET = set(' 　\t\n\xa0\r')
def strip_punct(s):
    return ''.join(c for c in s if c not in PUNCT_SET and c not in WS_SET)
def strip_punct_no_pua(s):
    return ''.join(c for c in s
                   if c not in PUNCT_SET and c not in WS_SET and not is_pua(c))

VARIANT_MAP = {
    # yitizi 漏掉的 14 个冷僻异体（其余 100 多个由 yitizi 自动覆盖）
    '衘':'衔','濓':'濂','兊':'兑','塜':'冢','贠':'员','冺':'泯','滛':'淫',
    '髠':'髡','兾':'冀','蝡':'蠕','畨':'番','桞':'柳','秪':'祇','歘':'欻',
}

from opencc import OpenCC
import heapq, functools
try:
    import yitizi
    _HAS_YITIZI = True
except ImportError:
    _HAS_YITIZI = False
_t2s = OpenCC('t2s')

def is_pua(c):
    return 0xE000 <= ord(c) <= 0xF8FF

# Reverse VARIANT_MAP so canonical-set lookup is symmetric:
# if VARIANT_MAP[x]=y then y should also know x.
_REV_VARIANT_MAP = {}
for _k, _v in VARIANT_MAP.items():
    _REV_VARIANT_MAP.setdefault(_v, set()).add(_k)

@functools.lru_cache(maxsize=50000)
def _variants(c):
    """Return the full equivalence class of c (transitively closed across yitizi
    + VARIANT_MAP forward/reverse + OpenCC t2s). Two chars are equivalent iff
    their variant sets share any element."""
    if is_pua(c):
        return frozenset()
    cands = {c}
    for _ in range(4):  # 4 iterations is enough for typical 异体字 chains
        new = set(cands)
        for x in cands:
            if _HAS_YITIZI:
                try:
                    new |= set(yitizi.get(x) or [])
                except Exception:
                    pass
            new.add(VARIANT_MAP.get(x, x))
            new |= _REV_VARIANT_MAP.get(x, set())
            new.add(_t2s.convert(x))
        if new == cands:
            break
        cands = new
    return frozenset(cands)

def char_eq(rc, oc):
    if is_pua(rc):
        return True
    if rc == oc:
        return True
    a = _variants(rc)
    if not a:
        return False
    return oc in a or bool(a & _variants(oc))

# ---- Layer 1: strict (no length diff) ----
def merge_strict(raw, out):
    raw_content = [c for c in raw if c not in PUNCT_SET and c not in WS_SET]
    ri = 0; result = []
    for oc in out:
        if oc in PUNCT_SET or oc in WS_SET:
            result.append(oc); continue
        if ri >= len(raw_content): return None
        rc = raw_content[ri]
        if char_eq(rc, oc):
            result.append(rc); ri += 1
        else:
            return None
    if ri != len(raw_content): return None
    if strip_punct(''.join(result)) != strip_punct(raw): return None
    return ''.join(result)

# ---- Layer 2: same-count force align ----
def merge_by_count(raw, out):
    raw_c = [c for c in raw if c not in PUNCT_SET and c not in WS_SET]
    out_c = [c for c in out if c not in PUNCT_SET and c not in WS_SET]
    if len(raw_c) != len(out_c): return None
    ri = 0; result = []
    for oc in out:
        if oc in PUNCT_SET or oc in WS_SET:
            result.append(oc)
        else:
            result.append(raw_c[ri]); ri += 1
    return ''.join(result)

# ---- Layer 3: cost-bounded DP alignment (PUA + variant + drop/extra/soft_match) ----
def align_dp(raw, out, max_cost):
    """Dijkstra alignment. Cost-0: match/punct/pua_*. Cost-1: soft_match/raw_drop/model_extra."""
    R, M = len(raw), len(out)
    INF = max_cost + 1
    dist = {(0, 0): 0}
    parents = {(0, 0): None}
    pq = [(0, 0, 0)]
    while pq:
        c, i, j = heapq.heappop(pq)
        if c > dist.get((i, j), INF): continue
        if i == R and j == M: break
        def relax(ni, nj, dc, content):
            nc = c + dc
            if nc > max_cost: return
            if nc < dist.get((ni, nj), INF):
                dist[(ni, nj)] = nc
                parents[(ni, nj)] = (i, j, content)
                heapq.heappush(pq, (nc, ni, nj))
        # punct/ws from out
        if j < M and (out[j] in PUNCT_SET or out[j] in WS_SET):
            relax(i, j+1, 0, out[j])
        # match
        if (i < R and j < M and not is_pua(raw[i])
                and out[j] not in PUNCT_SET and out[j] not in WS_SET
                and char_eq(raw[i], out[j])):
            relax(i+1, j+1, 0, raw[i])
        # pua_drop (PUA dropped from output; AI didn't substitute)
        if i < R and is_pua(raw[i]):
            relax(i+1, j, 0, '')
        # pua_sub: PUA + content → use AI's substitute
        if (i < R and j < M and is_pua(raw[i])
                and out[j] not in PUNCT_SET and out[j] not in WS_SET):
            relax(i+1, j+1, 0, out[j])
        # pua_collapse: PUA + real + content → use real raw
        if (i+1 < R and j < M and is_pua(raw[i]) and not is_pua(raw[i+1])
                and out[j] not in PUNCT_SET and out[j] not in WS_SET):
            relax(i+2, j+1, 0, raw[i+1])
        # soft_match (unknown variant, cost 1)
        if (i < R and j < M and not is_pua(raw[i])
                and out[j] not in PUNCT_SET and out[j] not in WS_SET):
            relax(i+1, j+1, 1, raw[i])
        # raw_drop_real (model dropped a real char; preserve it, cost 1)
        if i < R and not is_pua(raw[i]):
            relax(i+1, j, 1, raw[i])
        # model_extra (model added extra content; discard, cost 1)
        if j < M and out[j] not in PUNCT_SET and out[j] not in WS_SET:
            relax(i, j+1, 1, '')
    if (R, M) not in dist: return None
    path = []
    cur = (R, M)
    while parents.get(cur) is not None:
        pi, pj, content = parents[cur]
        path.append(content)
        cur = (pi, pj)
    path.reverse()
    return ''.join(path)

def truncate_to_raw(raw, out):
    """Cut hallucinated continuation after raw's content is covered."""
    raw_c = [c for c in raw if c not in PUNCT_SET and c not in WS_SET]
    if not raw_c: return out
    ri = 0; last_j = -1
    for j, oc in enumerate(out):
        if oc in PUNCT_SET or oc in WS_SET: continue
        if ri >= len(raw_c): break
        last_j = j; ri += 1
    if ri < len(raw_c): return out
    cut = last_j + 1
    while cut < len(out) and (out[cut] in PUNCT_SET or out[cut] in WS_SET):
        cut += 1
    return out[:cut]

def find_start_in_out(raw, out, window=12, min_ratio=0.7):
    """Find first index in out where next window chars match raw's prefix (for leak-at-start)."""
    raw_c = [c for c in raw if c not in PUNCT_SET and c not in WS_SET][:window]
    if len(raw_c) < window: window = len(raw_c)
    out_pos = [(j, c) for j, c in enumerate(out) if c not in PUNCT_SET and c not in WS_SET]
    if len(out_pos) < window: return 0
    thresh = int(window * min_ratio)
    for s in range(len(out_pos) - window + 1):
        ok = sum(1 for k in range(window)
                 if is_pua(raw_c[k]) or char_eq(raw_c[k], out_pos[s+k][1]))
        if ok >= thresh:
            return out_pos[s][0]
    return 0

def rescue_cascade(raw, out):
    """Returns (merged, layer_tag) or (None, None)."""
    r = merge_strict(raw, out)
    if r is not None and strip_punct_no_pua(r) == strip_punct_no_pua(raw):
        return r, 'strict'
    r = merge_by_count(raw, out)
    if r is not None and strip_punct_no_pua(r) == strip_punct_no_pua(raw):
        return r, 'count'
    for mc in [0, 1, 2, 3, 5, 8, 12, 15]:
        r = align_dp(raw, out, max_cost=mc)
        if r is not None: return r, f'dp_c{mc}'
    # Try trailing-leak truncation
    tr = truncate_to_raw(raw, out)
    if tr != out:
        r = align_dp(raw, tr, max_cost=15)
        if r is not None: return r, 'truncraw_dp'
    # Try leading-leak shift
    start = find_start_in_out(raw, out)
    if start > 0:
        sliced = out[start:]
        r = align_dp(raw, sliced, max_cost=15)
        if r is not None: return r, 'startshift_dp'
        sliced2 = truncate_to_raw(raw, sliced)
        r = align_dp(raw, sliced2, max_cost=15)
        if r is not None: return r, 'startshift_truncraw_dp'
    return None, None

STOP_STRINGS = ['\n\n无标点：', '\n无标点：', '<|im_end|>', '<|endoftext|>',
                '本朝六卿之设', '自古帝王之有天下', '奉天门常朝', '吾乡布衣沈先生']

def punct_one_attempt(raw, jitter=0.0, max_tokens_override=None):
    """单次调用模型加标点。
    A 改进：max_tokens 公式从 (len(raw)*2+100) 收紧到 (len(raw)*1.5+50)
    B 改进：retry 时由 max_tokens_override 控制（外部按首次输出长度 ×1.1 卡死）
    """
    prompt = FEWSHOT_PROMPT.format(raw=raw)
    if max_tokens_override is not None:
        max_tok = max_tokens_override
    else:
        max_tok = max(256, min(2000, int(len(raw) * 1.5 + 50)))
    r = llm(
        prompt,
        temperature=jitter,
        max_tokens=max_tok,
        stop=STOP_STRINGS,
        echo=False,
    )
    return r['choices'][0]['text'].strip()


# === 长段切分：让模型自己挑断句位置 ===
# 严格启发式：句末助词只取 也/矣/焉/哉，句首词只取 又/初/后/然/按
# 必须满足：raw[i-1] ∈ END_PARTICLES 且 raw[i] ∈ START_WORDS
# 且：位置之前 50 字内若有 曰/云，必须有句末助词在 曰/云 和 i 之间（保证引语已闭合）
END_PARTICLES = set('也矣焉哉')
START_WORDS = set('又初后然按')
QUOTE_OPENERS = set('曰云')

def _has_open_quote_context(raw, pos, look_back=50):
    """pos 之前 look_back 范围内若有未闭合的 曰/云，返回 True（这种位置不能切）"""
    lo = max(0, pos - look_back)
    for i in range(pos - 1, lo - 1, -1):
        if raw[i] in QUOTE_OPENERS:
            for j in range(i + 1, pos):
                if raw[j] in END_PARTICLES:
                    return False  # 已闭合
            return True  # 仍开着
    return False


SPLIT_QUERY_PROMPT = """从以下 {n} 个候选位置中选最适合作为句号的位置（古文断句习惯，| 表示候选切分点）：
{candidates}

答案（只输出 {letters} 中的一个字母）："""


def find_best_split(raw, target_pos, window=120):
    """在 [target-window, target+window] 找合规候选，至多 3 个，让模型选。无候选返回 -1。"""
    start = max(20, target_pos - window)
    end = min(len(raw) - 20, target_pos + window)
    valid = []
    for i in range(start, end):
        if i < 1 or i >= len(raw) - 1:
            continue
        if raw[i-1] in END_PARTICLES and raw[i] in START_WORDS:
            if not _has_open_quote_context(raw, i):
                valid.append(i)
    if not valid:
        return -1  # 找不到合规位置 → 不切
    if len(valid) == 1:
        return valid[0]
    # 取最靠近 target_pos 的 3 个
    valid.sort(key=lambda x: abs(x - target_pos))
    candidates = sorted(valid[:3])
    if len(candidates) == 1:
        return candidates[0]
    letters = 'ABC'[:len(candidates)]
    options = []
    for i, pos in enumerate(candidates):
        before = raw[max(0, pos-10):pos]
        after = raw[pos:pos+10]
        options.append(f"{letters[i]}. ...{before} | {after}...")
    prompt = SPLIT_QUERY_PROMPT.format(
        n=len(candidates), candidates='\n'.join(options), letters='/'.join(letters))
    try:
        r = llm(prompt, max_tokens=3, stop=['\n', '。', ' ', '\t'],
                temperature=0.0, echo=False)
        answer = r['choices'][0]['text'].strip().upper()
        for i, c in enumerate(letters):
            if c in answer:
                return candidates[i]
    except Exception:
        pass
    return candidates[0]  # 模型乱答时默认取最早一个


def split_long_raw(raw, max_chunk=500):
    """递归切分到每块 ≤ max_chunk 字。找不到合规切点就保持原长度（不切）。"""
    if len(raw) <= max_chunk:
        return [raw]
    mid = len(raw) // 2
    split_pos = find_best_split(raw, mid)
    if split_pos < 0 or split_pos < 50 or split_pos > len(raw) - 50:
        return [raw]  # 没合规切点 → 整段，让 _punct_one_core 自己应对（可能 gave_up）
    left, right = raw[:split_pos], raw[split_pos:]
    if max(len(left), len(right)) >= len(raw) - 10:
        return [raw]
    return split_long_raw(left, max_chunk) + split_long_raw(right, max_chunk)


def _punct_one_core(raw, max_retries=2):
    """单段标点核心，B 改进：retry 时 max_tokens 卡死在首次输出长度 × 1.1。"""
    first_out_len = None
    for attempt in range(max_retries):
        if attempt == 0:
            out = punct_one_attempt(raw, jitter=0.0)
            first_out_len = len(out)
        else:
            retry_cap = max(256, int(first_out_len * 1.1)) if first_out_len else None
            out = punct_one_attempt(raw, jitter=0.3, max_tokens_override=retry_cap)
        if strip_punct(raw) == strip_punct(out):
            return out, 'strict_ok', attempt + 1
        merged, layer = rescue_cascade(raw, out)
        if merged is not None:
            return merged, layer, attempt + 1
    return None, 'gave_up', max_retries


def punct_one(raw, max_retries=2):
    """对长段先切分，再分块处理；短段走原路径。
    部分块失败 → 用原文填进去 + 整段标记为 split_partial，整段仍写入主输出
    引号不平衡 → 整段标记为 split_imbalanced，仍写入主输出
    """
    if len(raw) > 500:
        chunks = split_long_raw(raw, max_chunk=500)
        if len(chunks) > 1:
            results = []
            failed = []
            for i, c in enumerate(chunks):
                out, status, tries = _punct_one_core(c, max_retries)
                if out is None:
                    results.append(c)
                    failed.append(i)
                else:
                    results.append(out)
            merged = ''.join(results)
            imbalanced = (merged.count('「') != merged.count('」')
                          or merged.count('《') != merged.count('》')
                          or merged.count('『') != merged.count('』'))
            tags = [f'split_ok_{len(chunks)}']
            if failed:
                tags.append(f'partial_failed_{failed}')
            if imbalanced:
                tags.append('imbalanced')
            status = '|'.join(tags) if len(tags) > 1 else tags[0]
            return merged, status, 1
    return _punct_one_core(raw, max_retries)



test = '朝廷每端午节赐朝官吃糕糭于午门外酒数行而出文职大臣仍从驾幸后苑观武臣射栁事毕'
out, status, tries = punct_one(test)
print(f'sample (status={status}, tries={tries}):\n  {out or "(放弃)"}')


## 6. 数据 + 断点续跑

In [ ]:
import json

if not os.path.exists(IN_PATH):
    print(f'{IN_PATH} 不存在，用 kagglehub 主动下载 dataset...')
    %pip install -q kagglehub
    import kagglehub
    ds_path = kagglehub.dataset_download(DATASET_SLUG)
    IN_PATH = f'{ds_path}/{BOOK}.jsonl'
    print(f'dataset 下载到 {ds_path}')

rows = [json.loads(l) for l in open(IN_PATH, 'r', encoding='utf-8') if l.strip()]
rows = [r for r in rows if r['raw'].strip()]
print(f'total rows: {len(rows)}')

# 切片：rows[PART-1::PART_OF] —— 跨步切，每份均匀分布在整本书各处（长短段都均匀）
my_rows = rows[PART-1::PART_OF]
print(f'本份 part {PART}/{PART_OF}: {len(my_rows)} 段')

done_ids = set()
if os.path.exists(CKPT):
    done_ids = set(json.load(open(CKPT)))
    print(f'resuming, already done: {len(done_ids)}')

todo = [r for r in my_rows if r['id'] not in done_ids]
print(f'todo: {len(todo)}')


## 7. 主循环（顺序单流）

In [ ]:
import time, traceback, json
from collections import Counter

PRINT_EVERY = 50   # 每 N 段打一次进度（避免 Kaggle 日志面板卡死）

out_f = open(OUT_PATH, 'a', encoding='utf-8')
FAIL_PATH = f'/kaggle/working/{BOOK}{SUFFIX}-failures.jsonl'
REVIEW_PATH = f'/kaggle/working/{BOOK}{SUFFIX}-needs-review.jsonl'
STATUS_PATH = f'/kaggle/working/{BOOK}{SUFFIX}-status.txt'
fail_f = open(FAIL_PATH, 'a', encoding='utf-8')
review_f = open(REVIEW_PATH, 'a', encoding='utf-8')
stats = Counter()

t0 = time.time()
for i, r in enumerate(todo):
    t_para = time.time()
    try:
        out, status, tries = punct_one(r['raw'])
    except Exception as e:
        stats['err'] += 1
        traceback.print_exc()
        json.dump(list(done_ids), open(CKPT, 'w'))
        continue

    dt = time.time() - t_para
    stats[status] += 1

    if status != 'gave_up':
        rec = {'id': r['id'], 'raw': r['raw'], 'punct': out, 'source': status, 'tries': tries}
        out_f.write(json.dumps(rec, ensure_ascii=False) + '\n'); out_f.flush()
        # 长段切分时部分块失败 or 引号不平衡 → 同时写到 review 文件，事后处理
        if 'partial_failed' in status or 'imbalanced' in status:
            review_f.write(json.dumps(rec, ensure_ascii=False) + '\n'); review_f.flush()
    else:
        last = punct_one_attempt(r['raw'], jitter=0.3)
        fail_rec = {'id': r['id'], 'raw': r['raw'], 'last_attempt': last, 'status': status}
        fail_f.write(json.dumps(fail_rec, ensure_ascii=False) + '\n'); fail_f.flush()
        out = last

    done_ids.add(r['id'])

    # 每 PRINT_EVERY 段或最后一段：打印进度 + 写 checkpoint + 写 status 文件
    if (i + 1) % PRINT_EVERY == 0 or i == len(todo) - 1:
        json.dump(list(done_ids), open(CKPT, 'w'))
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        eta_min = (len(todo) - i - 1) / rate / 60 if rate > 0 else 0
        total_ok = sum(v for k, v in stats.items() if k not in ('gave_up', 'err'))
        ok_pct = total_ok / (i + 1) * 100
        line = (f'[{i+1}/{len(todo)} {ok_pct:.1f}% rate={rate:.2f}/s '
                f'ETA={eta_min:.0f}m] '
                f'ok={total_ok} fail={stats["gave_up"]} err={stats["err"]} '
                f'| layers: ' + ' '.join(f'{k}={v}' for k, v in stats.most_common(4)))
        print(line)
        # 同步写到一个小文件，你随时下载/cat 就能看实时状态
        with open(STATUS_PATH, 'w') as sf:
            sf.write(line + '\n')

out_f.close()
fail_f.close()
review_f.close()
json.dump(list(done_ids), open(CKPT, 'w'))

total = sum(stats.values())
total_ok = sum(v for k, v in stats.items() if k not in ('gave_up', 'err'))
print(f'\nDONE — 成功率: {total_ok}/{total} ({total_ok/total*100:.1f}%)')
print('按层级分布:')
for k, v in sorted(stats.items(), key=lambda x: -x[1]):
    print(f'  {v:5d}  {k}')
print(f'\n输出: {OUT_PATH}\n失败: {FAIL_PATH}\n需复查: {REVIEW_PATH}\n状态: {STATUS_PATH}')


## 8. 打包 outputs 成 zip + 下载说明

跑完点顶栏 **Save Version → Quick Save** 才能在 Output 面板看到 zip 下载。

In [ ]:
import zipfile, os

zip_path = f'/kaggle/working/{BOOK}-outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(os.listdir('/kaggle/working/')):
        full = f'/kaggle/working/{f}'
        if os.path.isfile(full) and not f.endswith('.zip'):
            zf.write(full, f)
            print(f'  + {f} ({os.path.getsize(full)/1024:.1f} KB)')
print(f'\n打包完成: {zip_path}  ({os.path.getsize(zip_path)/1024:.1f} KB)')
print('\n--- /kaggle/working/ 全部文件 ---')
!ls -lh /kaggle/working/